In [3]:
# Cell 1: Imports & Setup
from pathlib import Path
import json
import shutil
import numpy as np
import pandas as pd
import polars as pl
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
import optuna
import gc

data_base_dir = Path('data/regionwise_yearwise_samples')
artifact_dir = Path('data/feature_selection_artifacts')
output_config_dir = Path('data/optuna_regional_gee_configs')
output_config_dir.mkdir(parents=True, exist_ok=True)

TARGET_REGION = 'DES'
REGION_DIR = data_base_dir / TARGET_REGION
BENCHMARK_YEARS = [1995, 2005, 2015, 2024]

cache_dir = Path(f'data/cache_splits_{TARGET_REGION}')
if cache_dir.exists():
    shutil.rmtree(cache_dir)
cache_dir.mkdir(parents=True, exist_ok=True)

# Load feature artifacts from 3a and 3b
with open(artifact_dir / f"{TARGET_REGION}_ranked_features.json", "r") as f:
    all_ranked = json.load(f)["ranked_features"]

with open(artifact_dir / f"{TARGET_REGION}_uncorrelated_features.json", "r") as f:
    uncorr_ranked = json.load(f)["uncorrelated_features"]

print(f"Loaded {len(all_ranked)} ranked features, {len(uncorr_ranked)} uncorrelated features.")

Loaded 104 ranked features, 36 uncorrelated features.


In [5]:
# Cell 2: Pre-Partition Benchmark Years into Compressed Split Caches
old_codes = [5, 3, 4, 12, 11, 66, 19, 36, 9, 41, 24, 23, 75, 30, 25, 33, 76, 34, 61]
new_codes = [5, 3, 4, 12, 11, 66, 19, 14, 14, 14, 22, 22, 22, 22, 22, 33, 76, 34, 61]
class_remap_dict = dict(zip(old_codes, new_codes))

print("Loading benchmark years individually from year partitions...")

for yr in BENCHMARK_YEARS:
    year_path = REGION_DIR / f"year={yr}"
    
    # Read ONLY this single year folder off disk
    df_yr = (
        pl.read_parquet(year_path / "*.parquet")
        .with_columns(
            pl.col('class').replace_strict(class_remap_dict, default=pl.col('class')).cast(pl.Int32)
        )
        .select(all_ranked + ['class'])
    )
    
    X_yr = df_yr.select(all_ranked).to_numpy().astype(np.float32)
    y_yr = df_yr['class'].to_numpy().astype(np.int32)
    
    del df_yr
    gc.collect()
    
    # 80:20 Stratified Split per year (locked validation fold)
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_yr, y_yr, test_size=0.20, stratify=y_yr, random_state=42
    )
    
    # Save compact binary split cache
    np.savez_compressed(
        cache_dir / f"split_{yr}.npz",
        X_tr=X_tr, y_tr=y_tr, X_val=X_val, y_val=y_val
    )
    print(f"  Cached year {yr}: Train={X_tr.shape[0]:,}, Val={X_val.shape[0]:,}")
    
    del X_yr, y_yr, X_tr, X_val, y_tr, y_val
    gc.collect()

print("Partitioning complete. Memory fully cleared.")

Loading benchmark years individually from year partitions...
  Cached year 1995: Train=93,876, Val=23,470
  Cached year 2005: Train=93,909, Val=23,478
  Cached year 2015: Train=93,909, Val=23,478
  Cached year 2024: Train=93,909, Val=23,478
Partitioning complete. Memory fully cleared.


In [6]:
# Cell 3: Reference Balancing Setup
feat_to_idx = {name: i for i, name in enumerate(all_ranked)}

ref_csv_path = Path('data/classwise_samples_reference_ratios.csv')
ref_df = pd.read_csv(ref_csv_path)
ref_df['old_code'] = old_codes
ref_df['new_code'] = new_codes
ref_region_cols = [c for c in ref_df.columns if c not in ['l2_code', 'l2_name', 'l1_parent', 'old_code', 'new_code']]
parent_ref_df = ref_df.groupby('new_code')[ref_region_cols].sum()
ref_col_series = parent_ref_df[TARGET_REGION] if TARGET_REGION in parent_ref_df.columns else None

def get_balanced_indices(y_train_arr, target_n, ref_series, rng):
    classes = np.unique(y_train_arr)
    col_present = ref_series.reindex(classes).fillna(0.0)
    scaled = (np.sqrt(col_present) / sum(np.sqrt(col_present))) * target_n
    selected = []
    for cls in classes:
        cls_idx = np.where(y_train_arr == cls)[0]
        desired = int(np.round(scaled.get(cls, 0)))
        n_sample = min(len(cls_idx), max(3, desired))
        selected.extend(rng.choice(cls_idx, size=n_sample, replace=False))
    return np.array(selected)

In [7]:
# Cell 4: Multi-Objective Optuna Study Loop
def objective(trial):
    sampling_strategy = trial.suggest_categorical(
        "sampling_strategy", ["stratified", "reference_balanced"]
    )
    
    feat_strategy = trial.suggest_categorical("feature_strategy", ["all", "uncorrelated"])
    if feat_strategy == "uncorrelated":
        active_names = uncorr_ranked
    else:
        num_bands = trial.suggest_int("max_input_features", 10, len(all_ranked))
        active_names = all_ranked[:num_bands]
        
    active_col_idx = [feat_to_idx[f] for f in active_names]
    train_fraction = trial.suggest_float("train_fraction", 0.20, 1.0, step=0.20)

    rf_params = {
        "n_estimators": trial.suggest_int("numberOfTrees", 50, 150, step=25),
        "min_samples_leaf": trial.suggest_int("minLeafPopulation", 2, 12),
        "max_features": "sqrt",
        "max_depth": 16,
        "n_jobs": 2,
        "random_state": 42
    }

    yearly_macro_f1s = []
    total_samples_trained = 0
    rng = np.random.RandomState(42)

    for yr in BENCHMARK_YEARS:
        # Load only this year's cached arrays
        data = np.load(cache_dir / f"split_{yr}.npz")
        X_tr = data['X_tr']
        y_tr = data['y_tr']
        X_val = data['X_val'][:, active_col_idx]
        y_val = data['y_val']

        target_n = int(len(X_tr) * train_fraction)

        if train_fraction >= 0.99 and sampling_strategy == "stratified":
            chosen_local = np.arange(len(X_tr))
        elif sampling_strategy == "reference_balanced" and ref_col_series is not None:
            chosen_local = get_balanced_indices(y_tr, target_n, ref_col_series, rng)
        else:
            chosen_local = []
            for cls in np.unique(y_tr):
                cls_indices = np.where(y_tr == cls)[0]
                n_take = max(3, int(target_n * len(cls_indices) / len(y_tr)))
                chosen_local.extend(rng.choice(cls_indices, size=min(len(cls_indices), n_take), replace=False))
            chosen_local = np.array(chosen_local)

        total_samples_trained += len(chosen_local)
        X_sub = X_tr[chosen_local][:, active_col_idx]
        y_sub = y_tr[chosen_local]

        clf = RandomForestClassifier(**rf_params)
        clf.fit(X_sub, y_sub)

        preds = clf.predict(X_val)
        yearly_macro_f1s.append(f1_score(y_val, preds, average="macro"))

        del data, X_tr, y_tr, X_val, y_val, X_sub, y_sub, clf
        gc.collect()

    return float(np.mean(yearly_macro_f1s)), total_samples_trained

study = optuna.create_study(study_name=f"rf_{TARGET_REGION}", directions=["maximize", "minimize"])
study.optimize(objective, n_trials=40, n_jobs=1)

[I 2026-10-02 21:25:17,002] A new study created in memory with name: rf_DES
[I 2026-10-02 21:25:48,161] Trial 0 finished with values: [0.9354234128968871, 75101.0] and parameters: {'sampling_strategy': 'stratified', 'feature_strategy': 'all', 'max_input_features': 86, 'train_fraction': 0.2, 'numberOfTrees': 75, 'minLeafPopulation': 5}.
[I 2026-10-02 21:25:58,316] Trial 1 finished with values: [0.9489102650224308, 75101.0] and parameters: {'sampling_strategy': 'stratified', 'feature_strategy': 'all', 'max_input_features': 18, 'train_fraction': 0.2, 'numberOfTrees': 50, 'minLeafPopulation': 6}.
[I 2026-10-02 21:28:01,103] Trial 2 finished with values: [0.9730233123727845, 300458.0] and parameters: {'sampling_strategy': 'stratified', 'feature_strategy': 'uncorrelated', 'train_fraction': 0.8, 'numberOfTrees': 150, 'minLeafPopulation': 3}.
[I 2026-10-02 21:31:20,273] Trial 3 finished with values: [0.971902771277363, 375603.0] and parameters: {'sampling_strategy': 'stratified', 'feature_stra

In [8]:
from optuna.visualization import plot_pareto_front

fig = plot_pareto_front(
    study,
    target_names=["Mean Macro-F1", "Total Samples Consumed"]
)
fig.update_layout(
    title=f"Multi-Objective Pareto Front - {TARGET_REGION}",
    width=900,
    height=550
)
fig.show()

# Optional: export interactive HTML
fig.write_html(output_config_dir / f"{TARGET_REGION}_pareto_frontier.html")

In [10]:
# Cell 5: Extract Best Trial & Export GEE Config
pareto_trials = study.best_trials
best_trial = max(pareto_trials, key=lambda t: t.values[0])

best_strategy = best_trial.params["feature_strategy"]
if best_strategy == "uncorrelated":
    selected_bands = uncorr_ranked
    num_bands = len(selected_bands)
else:
    num_bands = best_trial.params["max_input_features"]
    selected_bands = all_ranked[:num_bands]

region_summary = {
    "region": TARGET_REGION,
    "sampling_strategy": best_trial.params.get("sampling_strategy", "stratified"),
    "feature_strategy": best_strategy,
    "macro_f1": best_trial.values[0],
    "total_samples": best_trial.values[1],
    "train_fraction": best_trial.params["train_fraction"],
    "num_input_features": num_bands,
    "numberOfTrees": best_trial.params["numberOfTrees"],
    "variablesPerSplit": int(np.sqrt(num_bands)),
    "minLeafPopulation": best_trial.params["minLeafPopulation"],
    "selected_bands": selected_bands
}

config_path = output_config_dir / f"{TARGET_REGION}_best_params.json"
with open(config_path, "w") as f:
    json.dump(region_summary, f, indent=2)

# Remove the scratch folder
shutil.rmtree(cache_dir)

print(f"\n[Finished {TARGET_REGION}]")
print(f"  Best Feature Strategy:  {best_strategy} ({num_bands} features)")
print(f"  Macro-F1: {region_summary['macro_f1']:.4f} | Total Samples: {region_summary['total_samples']:,}")
print(f"  GEE configuration saved to -> {config_path}")

FileNotFoundError: [Errno 2] No such file or directory: 'data/cache_splits_DES'